In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from src import utils

In [3]:
from huggingface_hub import HfFolder, login

api_file = "/home/fre.gilad/source/llm-iml/HF_KEY.txt"
hf_token = utils.api_key_from_file(api_file)

HfFolder.save_token(hf_token)
login(token=hf_token)

In [4]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("/home/fre.gilad/source/llm-iml/data/HarmBench/harmful_behaviors.csv")
data = data.rename(columns={"goal": "prompt"})

# shuffle the data
utils.set_seed(42)
data = data.sample(frac=1).reset_index(drop=True)

In [5]:
split = int(0.5 * len(data))
ds_train = data.iloc[:split].copy()
ds_eval = data.iloc[split:].copy()

# ds_train = data.copy()
# ds_eval = data.copy()

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=25, shuffle=False)

In [6]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

Train dataset size: 100
Eval dataset size: 100


In [7]:
from src.eval.harmbench_evaluator import HarmBenchEvaluator
from src.eval.template_evaluator import TemplateEvaluator
from src.eval.strong_reject_evaluator import StrongRejectEvaluator
from gserve.configs import ServeConfig, LLMConfig


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60),
    # ),
    StrongRejectEvaluator(
        serve_config=ServeConfig(gpu_ids=[1], startup_timeout=10 * 60, client_timeout=60),
        binary_thresh=0.5,
    ),
    TemplateEvaluator(),
]

INFO 07-15 12:46:54 [__init__.py:244] Automatically detected platform cuda.


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

INFO 07-15 12:47:09 [vllm_service.py:153] Launching subprocess:
    /home/fre.gilad/source/llm-iml/.venv/bin/python /home/fre.gilad/source/llm-iml/.venv/lib/python3.12/site-packages/gserve/vllm_server.py --serve --model google/gemma-2b --host 127.0.0.1 --port 51565 --gpus 1 --llm_kwargs {"dtype": "bfloat16", "tokenizer_mode": "auto", "trust_remote_code": false, "seed": 0, "enforce_eager": false} --lora_path /home/fre.gilad/.cache/huggingface/hub/models--qylu4156--strongreject-15k-v1/snapshots/4bd893d32390d2cace4f067dc2e3ef5294fd78a2
INFO 07-15 12:50:34 [vllm_service.py:210] Server is healthy at http://127.0.0.1:51565/health
INFO 07-15 12:50:34 [vllm_service.py:402] Started 1 server(s) listening on 127.0.0.1:51565


In [8]:
import torch
from notebooks.utils import print_supported_models, load_model

torch.set_float32_matmul_precision("high")

print_supported_models()

Supported Models:
- Qwen/Qwen3-0.6B
- Qwen/Qwen2.5-0.5B-Instruct
- GraySwanAI/Llama-3-8B-Instruct-RR
- GraySwanAI/Mistral-7B-Instruct-RR
- Orenguteng/Llama-3-8B-Lexi-Uncensored
- meta-llama/Meta-Llama-3-8B-Instruct
- meta-llama/Llama-3.2-1B-Instruct
- meta-llama/Llama-2-7b-chat-hf
- lmsys/vicuna-7b-v1.5
- mistralai/Mistral-7B-Instruct-v0.3
- tiiuae/falcon-7b-instruct
- tiiuae/Falcon3-7B-Instruct
- mosaicml/mpt-7b-chat
- microsoft/Orca-2-7b
- microsoft/Phi-3-mini-4k-instruct
- microsoft/Phi-4-mini-instruct
- upstage/SOLAR-10.7B-Instruct-v1.0
- openchat/openchat-3.5-0106
- HuggingFaceH4/zephyr-7b-beta
- cais/zephyr_7b_r2d2
- google/gemma-2b-it
- google/gemma-2-2b-it
- google/gemma-3-1b-it
- ContinuousAT/Llama-2-7B-CAT
- apple/OpenELM-1_1B-Instruct


In [9]:
model, tokenizer = load_model("meta-llama/Llama-2-7b-chat-hf")

Model Config:
model_name: meta-llama/Llama-2-7b-chat-hf
device_map: cuda:0
torch_dtype: torch.bfloat16
hf_token: None


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_e

In [ ]:
# TODO: (low priority) compare float32 + mixed precision performance vs regular bfloat16 vs float16:
# the loss landscape looks different (more smooth) but the UAP performance when using bfloat16 remains the same

from torch import optim
from src.sample_attacks import SoftPrompt, PEZ
from src.univ_attacks.iml import IML
from src.adv_model import AdvModel
from src.initialize import Initializer
from src.activ_extractor import ActivationExtractor
from src.config import GenConfig, StopCriteria


adv_model = AdvModel(model=model, tokenizer=tokenizer, num_tokens=20)

Initializer.from_string(adv_model, "! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! !", strict=False)


# inner_attack = SoftPrompt(
#     adv_model,
#     optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
#     steps=10,
#     mixed_precision=False,
# )

def attack_builder(adv_model: AdvModel, epoch: int):
    return SoftPrompt(
        adv_model,
        optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
        steps=int(1.5 * (epoch + 1)),
        mixed_precision=False,
    )

# inner_attack = PEZ(
#     adv_model,
#     num_optim_tokens=15, 
#     num_steps=50,
# )

optimizer = optim.AdamW(adv_model.parameters(), lr=2e-2)

activ_extractor = ActivationExtractor(model, "lm_head", capture_output=False)

gen_config = GenConfig(
    max_length=512,
    do_sample=False,
    # remove_invalid_values=True,
)

iml_attack = IML(
    adv_model=adv_model,
    inner_attack=attack_builder,
    optimizer=optimizer,
    activ_extractor=activ_extractor,
    evaluators=evaluators,
    eval_freq=0.5,
    gen_config=gen_config,
    mixed_precision=False,
    skip_already_fooled=False,
    skip_failed_attacks=True,
    log_dir="logs",
)

stop = StopCriteria(max_epochs=10, max_time=60 * 60)

Initialized from text: '! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! ! !'
Embed Tokens: ['▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!', '▁!']
Embed Length: 20


In [ ]:
adv_model = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)
iml_attack.close()

Logging enabled. Saving logs to: logs/meta-llama/Llama-2-7b-chat-hf/num_tokens_20/IML/2025-07-15_12-53-07
ClearML Task: created new task id=80a20cbdaa044507904ab0907cbb097d
ClearML results page: https://app.clear.ml/projects/a5f3128511554e9cafdfee319f8a9323/experiments/80a20cbdaa044507904ab0907cbb097d/output/log


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# TODO: (low priority) make sure the model computation actually runs at the model dtype
adv_model.set_embeddings(iml_attack.best_embeds)
adv_model.discretize()
iml_attack.evaluate(adv_model=adv_model, dl_eval=dl_eval, evaluators=evaluators)

In [ ]:
preds = iml_attack.predict(adv_model, dl_eval, max_length=300)
dl_eval.set_column("response", preds)

for i in range(len(preds)):
    print(" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")